# ⚠️ Manejo de Excepciones aplicado al CRUD
## Programación I — UADE

---

En este notebook vamos a aprender a manejar errores en Python aplicándolo directamente al **Sistema de Recomendación Musical** que venimos construyendo (`_4_actividad_integradora_template.py` de la unidad 6_Transformaciones_filtros).

Recordemos las estructuras de datos del proyecto:

| Estructura | Tipo | Contenido |
|---|---|---|
| `TEMAS` | lista de diccionarios (estática) | `id_tema`, `tema`, `autor` |
| `usuarios` | lista de diccionarios (CRUD) | `id_usuario`, `nombre`, `apellido`, `email` |
| `ratings` | lista de diccionarios (relación) | `id_rating`, `id_usuario`, `id_tema`, `rating`, `fecha` |

> 💡 Las excepciones permiten que el programa **no se caiga** ante un error, sino que lo maneje de forma controlada y le informe al usuario qué ocurrió.

Al final del notebook vamos a aplicar lo mismo a la **práctica del robot Unitree** (`mi_tp02.py`).

### Datos de ejemplo

Ejecutá esta celda primero: carga los datos que usan todos los ejemplos del notebook.

In [ ]:
from datetime import date

temas = [
    {"id_tema": 1, "tema": "Dai Dai",        "autor": "Shakira"},
    {"id_tema": 2, "tema": "Dynamite",       "autor": "BTS"},
    {"id_tema": 3, "tema": "DTMF",           "autor": "Bad Bunny"},
    {"id_tema": 4, "tema": "Dont Start Now", "autor": "Dua Lipa"},
    {"id_tema": 5, "tema": "Positions",      "autor": "Ariana Grande"},
]

usuarios = [
    {"id_usuario": 1, "nombre": "Ana",   "apellido": "Gómez", "email": "ana@mail.com"},
    {"id_usuario": 2, "nombre": "Bruno", "apellido": "Díaz",  "email": "bruno@mail.com"},
]

ratings = [
    {"id_rating": 1, "id_usuario": 1, "id_tema": 3, "rating": 5, "fecha": date.today()},
]


def buscar_usuario_por_id(id_usuario):
    return next((usuario for usuario in usuarios if usuario["id_usuario"] == id_usuario))


def buscar_tema_por_id(id_tema):
    return next((tema for tema in temas if tema["id_tema"] == id_tema))

print(f"{len(temas)} temas, {len(usuarios)} usuarios, {len(ratings)} rating(s) cargados.")

---
## Bloque 1 — Tipos de error en Python

Antes de manejar excepciones, es importante reconocer los distintos tipos de error:

| Tipo | Cuándo ocurre | Ejemplo en el proyecto |
|---|---|---|
| **SyntaxError** | El código está mal escrito | Falta un `:` en el `for` de `buscar_tema_por_id` |
| **ValueError** | El valor no tiene el formato esperado | `int("abc")` al pedir el ID del usuario |
| **TypeError** | Operación sobre un tipo incorrecto | `"Rating: " + 5` (sumar texto y número) |
| **KeyError** | Clave inexistente en un diccionario | `usuario["mail"]` si la clave es `"email"` |
| **IndexError** | Índice fuera de rango en una lista | `TEMAS[10]` si hay solo 5 temas |

Los **SyntaxError** se detectan antes de ejecutar. Los demás ocurren en tiempo de ejecución y son los que necesitamos manejar con `try/except`.

### 1.1 — ValueError: el más común en ingreso de datos

En nuestro proyecto, `mostrar_usuario_por_id()` y `eliminar_usuario()` leen el ID así:

```python
id_usuario = int(input("ID del usuario: "))
```

Si el usuario escribe `"abc"`, Python lanza un `ValueError` y **el programa entero se cae**, aunque estemos en medio del menú.

In [ ]:
# Simulamos el error sin try/except
entrada = "abc"
id_usuario = int(entrada)   # ← ValueError: invalid literal for int() with base 10: 'abc'

### 1.2 — KeyError: acceso a clave inexistente en un diccionario

Pasa mucho cuando nos equivocamos en el nombre de una clave.

In [ ]:
usuario = usuarios[0]

# La clave correcta es "email", no "mail"
print(usuario["mail"])   # ← KeyError: 'mail'

### 1.3 — IndexError: posición inexistente en una lista

In [ ]:
# Hay 5 temas → índices válidos: 0 a 4
print(temas[10])   # ← IndexError: list index out of range

---
## Bloque 2 — try / except

La estructura básica para manejar excepciones:

```python
try:
    # código que puede fallar
except TipoDeError as e:
    # qué hacer si falla
finally:
    # se ejecuta siempre, haya error o no (opcional)
```

- El bloque `try` se ejecuta normalmente.
- Si ocurre un error, Python salta al `except` correspondiente (el resto del `try` **no** se ejecuta).
- El `finally` se ejecuta siempre, con o sin error.

### 2.1 — Protegiendo `mostrar_usuario_por_id()`

Esta es la versión de la actividad integradora, ahora protegida contra un ID con letras:

In [ ]:
def mostrar_usuario_por_id():
    print("\n--- Buscar usuario por id ---")
    entrada = input("ID del usuario: ")

    try:
        id_usuario = int(entrada)
    except ValueError:
        print("ID inválido, debe ser un número entero.")
        return          # ← salimos de la función, el menú sigue funcionando

    usuario = buscar_usuario_por_id(id_usuario)
    if not usuario:
        print(f"No se encontró ningún usuario con id {id_usuario}.")
        return

    print(f"[{usuario['id_usuario']}] {usuario['nombre']} {usuario['apellido']} - {usuario['email']}")


# Probá con: 1, 2, 99 y "abc"
mostrar_usuario_por_id()

### 2.2 — Capturar múltiples excepciones

Dentro de un mismo `try` pueden ocurrir errores de distinto tipo. Usamos varios `except` para manejar cada caso por separado.

Ejemplo: el usuario elige un tema **por su posición** en la lista que ve en pantalla.

In [ ]:
print("--- Temas disponibles ---")
for i in range(len(temas)):
    print(f"  {i}. {temas[i]['tema']} — {temas[i]['autor']}")

entrada = input("Elegí el número de tema: ")

try:
    indice = int(entrada)          # puede lanzar ValueError
    tema = temas[indice]           # puede lanzar IndexError
    print(f"Elegiste: {tema['tema']} de {tema['autor']}")
except ValueError:
    print("Error: tenés que ingresar un número entero.")
except IndexError:
    print(f"Error: número fuera de rango. Hay {len(temas)} temas (0 a {len(temas) - 1}).")

> 🤔 **Pregunta para pensar:** si ingresás `-1`, ¿qué tema aparece? ¿Por qué no salta el `IndexError`?

### 2.3 — Capturar cualquier excepción con `Exception`

En el menú principal podemos envolver cada opción con `except Exception as e`, para que **ningún error inesperado rompa el programa**, sea cual sea el tipo.

```python
# En main() — opción "2" del submenú de usuarios
try:
    mostrar_usuario_por_id()
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")
```

> ⚠️ Capturar `Exception` genérico está bien en el **nivel más alto** del programa (el menú).
> En funciones internas es mejor capturar el tipo específico (`ValueError`, `KeyError`…) para no ocultar errores que en realidad son bugs nuestros.

In [ ]:
def leer_ratings():
    for rating in ratings:
        usuario = buscar_usuario_por_id(rating["id_usuario"])
        tema = buscar_tema_por_id(rating["id_tema"])
        # Bug a propósito: la clave es "rating", no "valor"
        print(f"{usuario['nombre']} → {tema['tema']}: {rating['valor']}/5") # Reemplazar por rating


# Así lo llamaría el menú
try:
    leer_ratings()
except Exception as e:
    print(f"Ocurrió un error inesperado ({type(e).__name__}): {e}")

print("El menú sigue funcionando 👍")

### 2.4 — `finally`: lo que se ejecuta pase lo que pase

`finally` es ideal para tareas de "cierre": cerrar un archivo, cortar una conexión… o **frenar un robot** (lo vamos a ver al final).

In [ ]:
def eliminar_usuario_por_id(entrada):
    print(f"\nIntentando eliminar el usuario '{entrada}'...")
    try:
        id_usuario = int(entrada)
        usuario = buscar_usuario_por_id(id_usuario)
        usuarios.remove(usuario)    # si usuario es None → ValueError
        print("Usuario eliminado correctamente.")
    except ValueError:
        print("No se pudo eliminar: ID inválido o inexistente.")
    finally:
        print(f"Usuarios registrados: {len(usuarios)}")


eliminar_usuario_por_id("abc")   # error de conversión
eliminar_usuario_por_id("99")    # no existe
eliminar_usuario_por_id("2")     # OK

---
## Bloque 3 — Lanzar excepciones con `raise`

`raise` se usa para **generar una excepción a propósito** cuando una regla del sistema no se cumple.
Es la forma de comunicar errores desde las funciones internas hacia el `try/except` del nivel superior.

```python
raise ValueError("El rating debe estar entre 1 y 5.")
raise TypeError("El id_tema debe ser un número entero.")
raise Exception("Mensaje de error genérico.")
```

### 3.1 — `raise` cuando se agotan los intentos

En `crear_usuario()` usamos `while not validar_nombre(nombre)`: si el usuario nunca acierta, el programa queda **atrapado para siempre**.

Una alternativa es darle un máximo de intentos y, si los agota, lanzar un `ValueError`:

```python
for intento in range(MAX_INTENTOS):
    nombre = input("Nombre: ").strip()
    if validar_nombre(nombre):
        break
    print("El nombre solo puede contener letras y espacios.")
else:
    raise ValueError("Se superó el máximo de intentos para el nombre.")
```

> 💡 El `else` de un `for` se ejecuta **solo si el bucle terminó sin un `break`**.
> Es decir: si el usuario nunca ingresó un valor válido en los 3 intentos.

In [ ]:
def validar_nombre(nombre):
    nombre = nombre.strip()
    if not nombre:
        return False
    return nombre.replace(" ", "").isalpha()


def leer_nombre():
    MAX_INTENTOS = 3
    for intento in range(MAX_INTENTOS):
        nombre = input(f"Intento {intento + 1}/{MAX_INTENTOS} — Nombre: ").strip()
        if validar_nombre(nombre):
            return nombre
        print("El nombre solo puede contener letras y espacios.")
    else:
        raise ValueError("Se superó el máximo de intentos para el nombre.")


# Así lo usaría crear_usuario() desde el menú
try:
    nombre = leer_nombre()
    print(f"Nombre válido: {nombre}")
except ValueError as e:
    print(f"No se pudo crear el usuario: {e}")

### 3.2 — `raise` para proteger los datos: `registrar_rating()`

En la actividad integradora, `registrar_rating()` tenía estas **mejoras futuras** pendientes:

- validar que el usuario y el tema existan
- (y nosotros agregamos) validar que el rating esté entre 1 y 5

Las resolvemos con `raise`: la función **no imprime** el error, lo **lanza**, y quien la llama decide qué mostrar.

In [ ]:
def registrar_rating(id_usuario, id_tema, valor_rating):
    if buscar_usuario_por_id(id_usuario) is None:
        raise ValueError(f"No existe el usuario con id {id_usuario}.")

    if buscar_tema_por_id(id_tema) is None:
        raise ValueError(f"No existe el tema con id {id_tema}.")

    if valor_rating < 1 or valor_rating > 5:
        raise ValueError(f"El rating debe estar entre 1 y 5 (se recibió {valor_rating}).")

    nuevo_rating = {
        "id_rating": len(ratings) + 1,
        "id_usuario": id_usuario,
        "id_tema": id_tema,
        "rating": valor_rating,
        "fecha": date.today(),
    }
    ratings.append(nuevo_rating)


# Probamos varios casos: (id_usuario, id_tema, rating)
casos = [
    (1, 2, 4),     # OK
    (99, 2, 4),    # usuario inexistente
    (1, 10, 4),    # tema inexistente
    (1, 2, 7),     # rating fuera de rango
]

for id_usuario, id_tema, valor in casos:
    try:
        registrar_rating(id_usuario, id_tema, valor)
        print(f"✅ Rating registrado ({id_usuario}, {id_tema}, {valor})")
    except ValueError as e:
        print(f"❌ {e}")

print(f"\nRatings guardados: {len(ratings)}")